## はじめに

このノートブックでは、Cortex Agent、Snowflake Managed MCP Server、およびOAuth認証を設定します。

**主な処理内容:**
1. Semantic Viewの作成（Cortex Analyst用）
2. Cortex Agentの作成（Cortex Analyst + Cortex Searchツール）
3. Snowflake Managed MCP Serverの作成
4. OAuth認証の設定（Claude / ChatGPT用）

---

### 全体アーキテクチャ

```
MCPクライアント（Claude、ChatGPT等）
        │
        ▼ MCP Protocol（OAuth認証）
┌───────────────────────────────────────────────┐
│     Snowflake Managed MCP Server              │
│                     │                         │
│     ┌───────────────┴───────────────┐         │
│     ▼                               ▼         │
│  Cortex Agent                  (他のツール)    │
│     │                                         │
│  ┌──┴──────────────────────┐                  │
│  ▼                         ▼                  │
│ Cortex Analyst         Cortex Search          │
│ (Semantic View)        (4 Services)           │
│     │                      │                  │
│     ▼                      ▼                  │
│ 構造化データ            非構造化データ          │
│ (売上・顧客等)          (FAQ・SNS等)           │
└───────────────────────────────────────────────┘
```

**MCP（Model Context Protocol）とは:**
- AIエージェントがビジネスアプリケーションや外部データシステムと安全に連携するためのオープンソース標準
- Snowflake Managed MCP Serverを使うことで、インフラ構築なしでAIエージェントがSnowflakeのデータにアクセス可能

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
USE WAREHOUSE MCP_HANDSON_WH;
USE SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 1. 前提条件の確認

Part 1で作成したCortex Search Serviceが存在することを確認します。

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 2. Semantic Viewの作成

Cortex Analyst向けのセマンティックビューを作成します。

**Semantic Viewとは:**
- ビジネスメトリクスやエンティティの関係を定義
- 自然言語クエリ（Text-to-SQL）を可能にする
- Cortex Agentのツールとして利用可能

**対象テーブル:**
- dim_customers: 顧客マスタ
- dim_products: 商品マスタ
- fact_orders: 注文トランザクション
- fact_payments: 決済トランザクション
- fact_web_logs: Webアクセスログ

In [ ]:
-- ============================================================================
-- Semantic View の作成
-- 対象テーブル: dim_customers, dim_products, fact_orders, fact_payments, fact_web_logs
-- ============================================================================
CREATE OR REPLACE SEMANTIC VIEW MCP_HANDSON_DB.ANALYTICS_SCHEMA.EC_ANALYSIS_SEMANTIC_VIEW
    TABLES (
        customers AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.dim_customers PRIMARY KEY (customer_id),
        products AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.dim_products PRIMARY KEY (product_id),
        orders AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_orders PRIMARY KEY (order_id),
        payments AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_payments PRIMARY KEY (payment_id),
        web_logs AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_web_logs PRIMARY KEY (log_id)
    )
    RELATIONSHIPS (
        orders_to_customers AS orders (customer_id) REFERENCES customers,
        orders_to_products AS orders (product_id) REFERENCES products,
        payments_to_orders AS payments (order_id) REFERENCES orders
    )
    FACTS (
        orders.sales_amount AS total_amount,
        payments.paid_amount AS payment_amount
    )
    DIMENSIONS (
        customers.full_name AS CONCAT(last_name, first_name),
        products.name AS product_name,
        orders.date AS order_datetime
    )
    METRICS (
        orders.total_sales AS SUM(total_amount),
        orders.order_count AS COUNT(order_id)
    )
    COMMENT = 'ECサイトの売上・顧客・商品・Webログを分析するためのセマンティックビュー';

In [ ]:
-- ============================================================================
-- Semantic Viewの確認
-- ============================================================================
SHOW SEMANTIC VIEWS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 3. Cortex Agentの作成

Cortex AnalystとCortex Searchをツールとして持つエージェントを作成します。

**エージェント設定:**
- オブジェクト名: `MCP_ANALYTICS_AGENT`
- 表示名: `MCP分析エージェント`
- モデル: `auto`（最新のモデルが自動選択）

**含まれるツール:**

| ツール名 | 種別 | 対象 |
|---------|------|------|
| EC_Sales_Analysis | Cortex Analyst (Semantic View) | 売上・顧客・商品データ |
| FAQ_Search | Cortex Search | FAQドキュメント |
| Operation_Manual_Search | Cortex Search | 業務マニュアル |
| Voice_Log_Search | Cortex Search | 音声ログ要約 |
| SNS_Mention_Search | Cortex Search | SNS投稿 |
| data_to_chart | データ可視化 | チャート生成 |

**重要な設定ポイント:**
- `orchestration.budget`: タイムアウトとトークン制限を設定
- `execution_environment`: Cortex Analystでのクエリ実行環境を指定
- `tool_resources`: Cortex Searchは`name`キーで指定（`search_service`ではない）
- `instructions`: マルチライン形式（`|`）を避け、1行で記載

In [ ]:
-- ============================================================================
-- Cortex Agent の作成（全ツール含む）
-- ============================================================================
CREATE OR REPLACE AGENT MCP_ANALYTICS_AGENT
  COMMENT = 'ECサイトの売上・顧客・VoC分析を自然言語で行うエージェントです。'
  PROFILE = '{"display_name": "MCP分析エージェント", "color": "blue"}'
FROM SPECIFICATION $$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 120
    tokens: 32000

instructions:
  response: "丁寧語で回答し、金額は3桁区切り、パーセンテージは小数点第1位まで表示してください。"
  orchestration: "売上・注文・顧客・商品に関する数値分析はEC_Sales_Analysisを使用。FAQ関連はFAQ_Search、業務マニュアルはOperation_Manual_Search、過去の問い合わせはVoice_Log_Search、SNSの評判はSNS_Mention_Searchを使用してください。"
  system: "あなたはECサイト「GlacierStyle」の分析アシスタントです。売上・注文データは2024年のものです。"

tools:
  # Cortex Analyst（売上・顧客分析）
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: EC_Sales_Analysis
      description: "ECサイトの売上、注文、顧客、商品、決済データを分析します。売上推移、カテゴリ別売上、顧客セグメント分析、購買傾向などの質問に回答できます。"

  # Cortex Search（FAQドキュメント）
  - tool_spec:
      type: cortex_search
      name: FAQ_Search
      description: "ECサイトのよくある質問（FAQ）から回答を検索します。返品・交換、配送、支払い、会員登録などに関する質問に対応します。"

  # Cortex Search（業務マニュアル）
  - tool_spec:
      type: cortex_search
      name: Operation_Manual_Search
      description: "カスタマーサポート業務の運営マニュアルから手順や対応方法を検索します。クレーム対応、返品処理、エスカレーション手順などを参照できます。"

  # Cortex Search（音声ログ）
  - tool_spec:
      type: cortex_search
      name: Voice_Log_Search
      description: "コールセンターの過去の通話履歴（要約）から類似事例を検索します。過去の問い合わせ対応事例を参照できます。"

  # Cortex Search（SNS投稿）
  - tool_spec:
      type: cortex_search
      name: SNS_Mention_Search
      description: "SNS上の関連投稿から顧客の声を検索します。商品の評判、ブランドイメージ、改善要望などを参照できます。"

  # データ可視化ツール
  - tool_spec:
      type: data_to_chart
      name: data_to_chart
      description: "データからチャートを生成します。"

tool_resources:
  # Cortex Analystの設定（execution_environmentが必須）
  EC_Sales_Analysis:
    execution_environment:
      type: warehouse
      warehouse: "MCP_HANDSON_WH"
    semantic_view: MCP_HANDSON_DB.ANALYTICS_SCHEMA.EC_ANALYSIS_SEMANTIC_VIEW

  # FAQ検索の設定（nameキーを使用）
  FAQ_Search:
    name: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_FAQ
    max_results: 5

  # 業務マニュアル検索の設定
  Operation_Manual_Search:
    name: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_OPERATION_MANUALS
    max_results: 5

  # 音声ログ検索の設定
  Voice_Log_Search:
    name: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_VOICE_LOGS
    max_results: 5

  # SNS投稿検索の設定
  SNS_Mention_Search:
    name: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_SNS_MENTIONS
    max_results: 10
$$;

In [ ]:
-- ============================================================================
-- 作成したエージェントの確認
-- ============================================================================
SHOW AGENTS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- エージェントの詳細確認
-- ============================================================================
DESCRIBE AGENT MCP_ANALYTICS_AGENT;

## 4. Snowflake Managed MCP Serverの作成

Cortex AgentをMCPクライアントから呼び出せるようにするMCP Serverを作成します。

**MCP Serverとは:**
- Model Context Protocol (MCP) に準拠したサーバー
- AIエージェント（Claude Desktop、Cursor等）がSnowflakeのデータに安全にアクセスするためのインターフェース
- OAuth認証またはPAT認証に対応

**サポートされるツールタイプ:**
- `CORTEX_AGENT_RUN`: Cortex Agentをツールとして呼び出し
- `CORTEX_ANALYST_MESSAGE`: Cortex Analystを直接呼び出し
- `CORTEX_SEARCH_SERVICE_QUERY`: Cortex Searchを直接呼び出し
- `SYSTEM_EXECUTE_SQL`: SQLクエリを実行
- `GENERIC`: UDFやストアドプロシージャを呼び出し

**重要: descriptionの設定について**

MCPクライアント（Cursor、Claude Code等）は、`description`を参照してどのツールを呼び出すべきか判断します。  
descriptionには以下を含めることを推奨します：

1. **対象データ/システム**: 何を分析・検索できるか
2. **機能一覧**: 具体的にどのような分析が可能か
3. **ユースケース例**: どのような質問に回答できるか
4. **データの期間**: いつのデータが含まれているか

descriptionが曖昧だと、AIエージェントがツールを適切に選択できず、呼び出されない可能性があります。

In [ ]:
-- ============================================================================
-- Snowflake Managed MCP Server の作成
-- Cortex Agentをツールとして公開
-- ============================================================================
CREATE OR REPLACE MCP SERVER MCP_ANALYTICS_SERVER
  FROM SPECIFICATION $$
    tools:
      - title: "GlacierStyle EC Analytics Agent"
        name: "ec-analytics-agent"
        type: "CORTEX_AGENT_RUN"
        identifier: "MCP_HANDSON_DB.ANALYTICS_SCHEMA.MCP_ANALYTICS_AGENT"
        description: "ECサイト「GlacierStyle」のデータを分析するAIエージェントです。以下の分析が可能です: (1)売上分析: 総売上額、月別売上推移、商品別売上TOP10、顧客セグメント分析、購買傾向分析 (2)FAQ検索: 返品ポリシー、配送料、支払い方法、会員登録に関する質問への回答 (3)業務マニュアル検索: クレーム対応手順、返品処理フロー、エスカレーション手順 (4)過去の問い合わせ検索: コールセンターの通話履歴から類似事例を検索 (5)SNS分析: Twitter/Instagramでの商品評判、顧客の声、改善要望の検索。データは2024年のものです。"
  $$
;

In [ ]:
-- ============================================================================
-- MCP Serverの確認
-- ============================================================================
SHOW MCP SERVERS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- MCP Serverの詳細確認
-- ============================================================================
DESCRIBE MCP SERVER MCP_ANALYTICS_SERVER;

## 5. OAuth認証の設定

Claude や ChatGPT などのMCPクライアントからSnowflakeに接続するため、OAuth認証を設定します。

**OAuth認証の流れ:**
1. Snowflake側でSecurity Integrationを作成（Claude用 / ChatGPT用）
2. Client IDとClient Secretを取得
3. MCPクライアントで認証情報を設定
4. 初回接続時にブラウザでSnowflakeにログイン

**重要:**  
- Snowflake Managed MCP ServerはDynamic Client Registration (DCR) をサポートしていないため、事前にSecurity Integrationを作成する必要があります
- Claude と ChatGPT ではリダイレクトURIが異なるため、それぞれ別のSecurity Integrationを作成します

In [ ]:
-- ============================================================================
-- OAuth Security Integration の作成（Claude用）
-- 参考: https://zenn.dev/snowflakejp/articles/with-managed-mcp-server
-- ============================================================================
CREATE OR REPLACE SECURITY INTEGRATION CLAUDE_OAUTH
    TYPE = OAUTH
    OAUTH_CLIENT = CUSTOM
    OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
    OAUTH_REDIRECT_URI = 'https://claude.ai/api/mcp/auth_callback'
    ENABLED = TRUE
    COMMENT = 'Claude (Desktop/Web) 用のOAuth認証';

In [ ]:
-- ============================================================================
-- OAuth Security Integration の作成（ChatGPT用）
-- ============================================================================
CREATE OR REPLACE SECURITY INTEGRATION CHATGPT_OAUTH
    TYPE = OAUTH
    OAUTH_CLIENT = CUSTOM
    OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
    OAUTH_REDIRECT_URI = 'https://chatgpt.com/connector_platform_oauth_redirect'
    ENABLED = TRUE
    COMMENT = 'ChatGPT (Web) 用のOAuth認証';

### OAuth Client ID / Client Secret の取得

上記で作成した Security Integration から、OAuth認証に必要なClient ID / Client Secretを取得します。

In [ ]:
-- ============================================================================
-- Claude用 OAuth Client ID / Client Secret の取得
-- ============================================================================
SELECT 
    'Claude' AS client,
    PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CLAUDE_OAUTH')):OAUTH_CLIENT_ID::STRING AS oauth_client_id,
    PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CLAUDE_OAUTH')):OAUTH_CLIENT_SECRET::STRING AS oauth_client_secret;

In [ ]:
-- ============================================================================
-- ChatGPT用 OAuth Client ID / Client Secret の取得
-- ============================================================================
SELECT 
    'ChatGPT' AS client,
    PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CHATGPT_OAUTH')):OAUTH_CLIENT_ID::STRING AS oauth_client_id,
    PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CHATGPT_OAUTH')):OAUTH_CLIENT_SECRET::STRING AS oauth_client_secret;

### ⚠️ 重要: OAuth認証情報の保存

上記のコマンドで取得した `OAUTH_CLIENT_ID` と `OAUTH_CLIENT_SECRET` を安全な場所に保存してください。

これらの値は、Claude / ChatGPT の設定で使用します。

## 6. MCPクライアントの設定

作成したMCP ServerにMCPクライアントから接続する方法を説明します。

---

### 6-1. エンドポイントURLの確認

MCP Serverのエンドポイントは以下の形式です：

```
https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER
```

**<account_url>の形式:**
- `<orgname>-<account_name>.snowflakecomputing.com`

例: `myorg-myaccount.snowflakecomputing.com`

> ⚠️ **重要**: ホスト名にアンダースコア（`_`）が含まれている場合、MCP接続で問題が発生します。  
> アンダースコアはハイフン（`-`）に置き換えてください。  
> 例: `my_org-my_account` → `my-org-my-account`

In [ ]:
-- ============================================================================
-- アカウントURL情報の確認
-- 注意: ホスト名にアンダースコア(_)が含まれるとMCP接続で問題が発生するため、
--       ハイフン(-)に置き換える必要があります
-- ============================================================================
SELECT CURRENT_ORGANIZATION_NAME() AS org_name,
       CURRENT_ACCOUNT_NAME() AS account_name,
       REPLACE(
           UPPER(CURRENT_ORGANIZATION_NAME()) || '-' || UPPER(CURRENT_ACCOUNT_NAME()) || '.snowflakecomputing.com',
           '_',
           '-'
       ) AS account_url;

### 6-2. Claude での設定

Claude（デスクトップ版 / Web版）ではOAuth認証を使用してMCPサーバーに接続します。

参考: [Snowflake-managed MCP Server の OAuth 接続を Claude, ChatGPT で試してみる](https://zenn.dev/snowflakejp/articles/with-managed-mcp-server)

**設定手順:**

1. Claude の画面左下をクリックして「設定」を開く
2. 「コネクタ」→「カスタムコネクタを追加」を選択
3. 以下の情報を入力：

| 設定項目 | 値 |
|---------|----|
| 名前 | `Snowflake MCP` （任意） |
| 説明 | `EC分析エージェント` （任意） |
| URL | `https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER` |

4. 「詳細設定」を開き、OAuth認証情報を入力：

| 設定項目 | 値 |
|---------|----|
| OAuth クライアントID | Step 5で取得した `OAUTH_CLIENT_ID`（CLAUDE_OAUTH用） |
| クライアントキー | Step 5で取得した `OAUTH_CLIENT_SECRET`（CLAUDE_OAUTH用） |

5. 「追加」をクリック

> ⚠️ `<account_url>` はアンダースコア（`_`）をハイフン（`-`）に置き換えてください。

**使用方法:**

チャット画面で追加したコネクタを選択して質問すると、Snowflakeのデータにアクセスできます。

> **Note**: 初回接続時はブラウザでSnowflakeにログインする必要があります。

### 6-3. ChatGPT での設定

ChatGPT（Web版）でMCPサーバーを利用するには、「開発者モード」を有効にしてアプリを登録します。

参考: [Snowflake-managed MCP Server の OAuth 接続を Claude, ChatGPT で試してみる](https://zenn.dev/snowflakejp/articles/with-managed-mcp-server)

> **Note**: ChatGPT無料版ではアプリの登録はできますが、利用には有料プラン（Pro/Plus）が必要です。

**設定手順:**

1. ChatGPT Web版にログイン
2. 左下のユーザー名部分をクリックし、「設定」を開く
3. 「アプリ」を押し、「高度な設定」をクリック
4. 「開発者モード」をオンにする
5. 「アプリを作成する」ボタンをクリック
6. 以下の情報を入力：

| 設定項目 | 値 |
|---------|----|
| 名前 | `Snowflake MCP` （任意） |
| 説明 | `EC分析エージェント` （任意） |
| URL | `https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER` |
| OAuth クライアントID | Step 5で取得した `OAUTH_CLIENT_ID`（CHATGPT_OAUTH用） |
| クライアントキー | Step 5で取得した `OAUTH_CLIENT_SECRET`（CHATGPT_OAUTH用） |

> ⚠️ `<account_url>` はアンダースコア（`_`）をハイフン（`-`）に置き換えてください。

7. 「理解した上で、続行します」にチェック
8. 「作成する」をクリック

**使用方法:**

1. チャット画面の「+」ボタンをクリック
2. 登録したアプリを選択
3. 初回接続時はブラウザでSnowflakeにログイン
4. 質問を入力（例：「2024年12月の売上上位10商品を教えて」）

> **Enterprise向け**: Business/Enterpriseプランでは、管理者がアプリを許可することでワークスペース全体で利用可能になります。

### 6-4. 接続テスト用の質問例

MCPクライアントから以下のような質問を試してみてください：

**売上分析（Semantic View経由）:**
- 「2024年12月の売上上位10商品を教えて」
- 「カテゴリ別の売上構成比を教えて」
- 「新規顧客とリピーターの購買金額の違いは？」

**FAQ検索（Cortex Search経由）:**
- 「返品ポリシーについて教えて」
- 「配送料について教えて」

**マニュアル検索（Cortex Search経由）:**
- 「クレーム対応の手順を教えて」
- 「返品処理の方法は？」

**音声ログ検索（Cortex Search経由）:**
- 「配送遅延に関する過去の問い合わせを検索して」
- 「ネガティブな問い合わせの事例を教えて」

**SNS分析（Cortex Search経由）:**
- 「SNSでの商品の評判を教えて」
- 「Twitterでのポジティブな投稿を検索して」

## 7. クリーンアップ（オプション）

ハンズオン終了後、作成したリソースを削除する場合は以下を実行してください。

In [ ]:
-- ============================================================================
-- クリーンアップ: OAuth Integrationの削除
-- ============================================================================
-- DROP SECURITY INTEGRATION CLAUDE_OAUTH;
-- DROP SECURITY INTEGRATION CHATGPT_OAUTH;

In [ ]:
-- ============================================================================
-- クリーンアップ: 作成したオブジェクトの削除
-- ============================================================================
-- DROP MCP SERVER MCP_ANALYTICS_SERVER;
-- DROP AGENT MCP_ANALYTICS_AGENT;
-- DROP SEMANTIC VIEW EC_ANALYSIS_SEMANTIC_VIEW;
-- DROP CORTEX SEARCH SERVICE SEARCH_FAQ;
-- DROP CORTEX SEARCH SERVICE SEARCH_OPERATION_MANUALS;
-- DROP CORTEX SEARCH SERVICE SEARCH_VOICE_LOGS;
-- DROP CORTEX SEARCH SERVICE SEARCH_SNS_MENTIONS;
-- DROP DATABASE MCP_HANDSON_DB;

## まとめ

このノートブックでは、Snowflake Managed MCP Serverを構築しました。

### 作成したオブジェクト

| オブジェクト | 名前 | 説明 |
|------------|------|------|
| Semantic View | EC_ANALYSIS_SEMANTIC_VIEW | 売上・顧客・商品データの分析用 |
| Cortex Agent | MCP_ANALYTICS_AGENT | Semantic View + Cortex Search を統合 |
| MCP Server | MCP_ANALYTICS_SERVER | MCPクライアントからのアクセスポイント |
| OAuth Integration | CLAUDE_OAUTH | Claude用OAuth認証 |
| OAuth Integration | CHATGPT_OAUTH | ChatGPT用OAuth認証 |

### アーキテクチャのポイント

- **MCP Server**: MCPプロトコルに準拠したエンドポイントを提供
- **Cortex Agent**: 構造化データ（Semantic View）と非構造化データ（Cortex Search）を統合
- **OAuth認証**: Claude / ChatGPT からのセキュアなアクセスを実現

### Cortex Agent作成時の重要な設定ポイント

MCP Server経由でCortex Agentを正しく動作させるために、以下の点に注意してください：

| 設定項目 | 正しい設定 | 誤った設定（動作しない） |
|---------|-----------|----------------------|
| orchestration.budget | `seconds`と`tokens`を指定 | 未設定 |
| instructions | 1行形式で記載 | マルチライン形式（`\|`）を使用 |
| tool_resources (Cortex Analyst) | `execution_environment`を含める | `execution_environment`なし |
| tool_resources (Cortex Search) | `name`キーで指定 | `search_service`キーで指定 |
| data_to_chart | ツールとして追加 | 未追加 |

### MCP Server作成時の重要な設定ポイント

- **description**: MCPクライアントがツールを適切に選択できるよう、具体的な機能一覧・対象データ・ユースケース例を記載
- **title**: 簡潔でわかりやすい名前を設定
- **identifier**: Cortex Agentの完全修飾名を正確に指定

### 次のステップ

- MCPクライアント（Cursor、Claude Code、ChatGPT Enterprise）から接続して分析を実行
- 追加のツール（UDF、ストアドプロシージャ等）をMCP Serverに登録
- 本番環境向けにSecurity Integrationのスコープやロールを調整